# 01 · Exploración de los datos sintéticos de Guavio
Relación clima → aportes → embalse → oferta y señales de condición de máquina. Ejecute antes `python scripts/run_pipeline.py`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
from src.data.loader import load_data
df = load_data()
df.describe().T.round(2)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
df.set_index('fecha')['nino34'].plot(ax=ax[0], title='Niño 3.4 (°C)')
df.set_index('fecha')['aportes_gwh'].rolling(30).mean().plot(ax=ax[1], title='Aportes, media 30 días (GWh/día)')
df.set_index('fecha')['volumen_util_pct'].plot(ax=ax[2], title='Volumen útil (%)')
plt.tight_layout()

In [ ]:
mo = df.set_index('fecha')[['nino34', 'aportes_pct_media']].resample('MS').mean()
pd.Series({f'{l} meses': mo['aportes_pct_media'].corr(mo['nino34'].shift(l)) for l in range(7)}).plot.bar(title='Correlación Niño 3.4 rezagado vs anomalía de aportes')

In [ ]:
d = df.assign(y=df.precio_oferta_cop_kwh / df.precio_minimo_cop_kwh)
d.plot.scatter(x='volumen_util_pct', y='y', c='nino34', cmap='coolwarm', alpha=0.4, figsize=(7, 4), title='Oferta normalizada vs volumen útil')

In [ ]:
import json
print(json.dumps(json.load(open('../reports/metrics/oferta_metrics.json'))['diebold_mariano_E2_vs_E1'], indent=2))
pd.read_csv('../reports/metrics/aportes_metrics_por_horizonte.csv').pivot(index='horizonte', columns='split', values='NSE')